In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../../archive/final_dataset_with_all_features.csv')

selected_columns = [
    'url_len', 'digits', 'letters', 'domain_ngram_entropy',
    'path_depth', 'path_entropy',
    'consonant_ratio', 'vowel_ratio', 'digit_ratio', 'avg_token_length',
    'token_count', 'label'
]

df = df[selected_columns]

X = df.drop('label', axis=1)
y = df['label']

print(f"\nDataset shape: {df.shape}")
print(f"Class distribution:\n{y.value_counts()}")
print(f"\nFeature names: {X.columns.tolist()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Ada Boost': AdaBoostClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')

    results[name] = model
    print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"Test accuracy: {(y_pred == y_test).mean():.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

In [ ]:
import joblib
for k,v in results.items():
    joblib.dump(v, k + ".pkl")